# MMM Phase 3 — Model Fitting & Channel Sensitivity

Fit PyMC-Marketing MMM on the Robyn dataset and analyze channel contributions.

In [ ]:
import sys, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import arviz as az

sys.path.insert(0, str(pathlib.Path().resolve().parent / 'src'))

from data_loader import load_raw
from model import build_mmm, fit_mmm
from sensitivity import channel_sensitivity, budget_reallocation

DATA_PATH = pathlib.Path().resolve().parent / 'data' / 'raw' / 'de_simulated_data.csv'
IDATA_PATH = pathlib.Path().resolve().parent / 'data' / 'interim' / 'mmm_idata.nc'

df = load_raw(DATA_PATH)
print(f'Loaded {len(df)} rows, columns: {list(df.columns)}')
df.head()

In [ ]:
mmm = build_mmm()
print('MMM model built:')
print(f'  Channel columns : {mmm.channel_columns}')
print(f'  Date column     : {mmm.date_column}')
print(f'  Adstock         : {mmm.adstock}')
print(f'  Saturation      : {mmm.saturation}')

In [ ]:
# NOTE: First run takes ~10 min on M1. Human approval required before executing this cell.
# Cached result is stored at data/interim/mmm_idata.nc for subsequent runs.

if IDATA_PATH.exists():
    print('Loading cached InferenceData from', IDATA_PATH)
    idata = az.from_netcdf(str(IDATA_PATH))
    mmm.idata = idata
else:
    print('Fitting MMM via MCMC (tune=500, draws=500, chains=2) — ~10 min on M1 ...')
    idata = fit_mmm(mmm, df)
    idata.to_netcdf(str(IDATA_PATH))
    print(f'Saved InferenceData to {IDATA_PATH}')

print('\nDivergences:', idata.sample_stats.diverging.values.sum())
print(az.summary(idata, var_names=['intercept']))

In [ ]:
# Posterior distributions
az.plot_posterior(idata, var_names=['intercept'])
plt.suptitle('Intercept posterior', y=1.02)
plt.tight_layout()
plt.show()

for param in ['alpha', 'lam']:
    mmm.plot_channel_parameter(param)
    plt.suptitle(f'Channel parameter: {param}', y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# Channel contributions
sensitivity = channel_sensitivity(mmm, df)
print('Channel sensitivity (posterior mean):')
print(sensitivity.to_string())

ax = sensitivity['mean_contribution'].plot.bar(yerr=sensitivity['std_contribution'],
                                                capsize=4, color=['steelblue', 'coral'])
ax.set_title('Mean channel contribution (original scale)')
ax.set_ylabel('Revenue contribution')
ax.set_xlabel('Channel')
plt.tight_layout()
plt.show()

In [ ]:
# Budget reallocation
realloc = budget_reallocation(mmm, df)
print('Budget reallocation analysis:')
print(realloc.to_string())

x = np.arange(len(realloc))
width = 0.35
fig, ax = plt.subplots()
ax.bar(x - width/2, realloc['actual_pct'], width, label='Actual %', color='steelblue')
ax.bar(x + width/2, realloc['optimal_pct'], width, label='Optimal %', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(realloc.index)
ax.set_ylabel('Budget allocation (%)')
ax.set_title('Actual vs. Contribution-Implied Budget Allocation')
ax.legend()
plt.tight_layout()
plt.show()

## Summary

### Key Findings
- Channel x1 and x2 posterior contributions are computed via `channel_contribution_forward_pass`.
- Budget reallocation delta shows which channel is under/over-funded relative to its contribution.
- Geometric adstock (l_max=8) captures carryover effects up to 8 weeks.
- Logistic saturation models diminishing returns per channel.

### Model Diagnostics
- Check divergence count above (should be < 1% of draws).
- R-hat values should be < 1.01 for all parameters.
- ESS bulk/tail should be > 400.

### Next Steps
- Phase 4: Stan model comparison (same data, different backend).
- Sensitivity analysis: vary adstock `l_max` and compare ELPD.
- Budget optimization using `mmm.optimize_budget()`.